# Pochettes AR : génération des modèles 3D par IA (Google Colab)

Pour chaque pochette du dossier `albums/` du projet :

1. **détourage** du sujet principal avec `rembg` (réseau U²-Net) ;
2. **reconstruction 3D** à partir de cette seule image avec **TripoSR** (Stability AI / Tripo, 2024) :
   un transformeur prédit une représentation 3D implicite (« triplan »), d'où on extrait un maillage
   coloré par *marching cubes* ;
3. **simplification** du maillage à ~3000 triangles (décimation par erreur quadrique, Garland & Heckbert 1997),
   pour que notre rasteriseur reste fluide ;
4. passage dans **le repère du projet** (Y vers le haut, posé sur la pochette, unité = largeur de la pochette)
   et écriture en `.obj` avec une couleur par sommet (`v x y z r g b`) ;
5. tout est rangé dans `cache/`, téléchargé en `cache.zip`, à décompresser à la racine du projet.

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*.
Compter ~10 min (installation surtout), puis ~10 s par pochette.

*Pourquoi Colab :* TripoSR a besoin d'une carte NVIDIA (CUDA). TRELLIS (Microsoft), plus précis, demande
au moins 16 Go de mémoire graphique : trop pour la T4 gratuite (15 Go).

In [ ]:
# 0. Vérifier qu'on a bien un GPU NVIDIA.
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Installation de TripoSR et des dépendances

In [ ]:
%cd /content
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd /content/TripoSR
!pip install -q omegaconf einops transformers huggingface-hub imageio trimesh scikit-image "rembg[gpu]" onnxruntime-gpu fast-simplification

TripoSR importe `torchmcubes` (marching cubes compilé en CUDA), dont la compilation échoue sur Colab.
Il n'en utilise qu'une fonction : on la remplace par celle de scikit-image (sur CPU, quelques secondes).
Convention à respecter : torchmcubes renvoie les sommets dans l'ordre (k, j, i) des indices du volume
(TripoSR les remet ensuite en (i, j, k)), alors que scikit-image les renvoie en (i, j, k).

In [ ]:
%%writefile /content/TripoSR/torchmcubes.py
"""Remplaçant minimal de torchmcubes, sur la base de skimage.measure.marching_cubes."""
import numpy as np
import torch
from skimage.measure import marching_cubes as _marching_cubes


def marching_cubes(volume, seuil):
    sommets, faces, _, _ = _marching_cubes(volume.detach().cpu().numpy().astype(np.float32), seuil)
    sommets = np.ascontiguousarray(sommets[:, ::-1])  # (i, j, k) -> (k, j, i), comme torchmcubes
    return torch.from_numpy(sommets).float(), torch.from_numpy(faces.astype(np.int64))

## 2. Récupération des pochettes du projet

In [ ]:
import json, shutil
from pathlib import Path

DEPOT = "https://github.com/Nowehh/projet_pochettes_ar.git"
!rm -rf /content/projet && git clone -q --depth 1 {DEPOT} /content/projet
ALBUMS = Path("/content/projet/albums")
# Sans accès au dépôt : créer /content/projet/albums à la main, y glisser les images et albums.json.

albums = json.loads((ALBUMS / "albums.json").read_text())
CACHE = Path("/content/cache")
shutil.rmtree(CACHE, ignore_errors=True)
(CACHE / "apercus").mkdir(parents=True)
for id_album, a in albums.items():
    print(f"{id_album:40s} {a['artiste']} - {a['album']}")

## 3. Réglages

- `TRIANGLES` : nombre de triangles visé après simplification ;
- `TAILLE` : plus grande dimension du modèle, en largeurs de pochette ;
- `RESOLUTION` : grille du marching cubes (plus fin = plus de détails avant simplification) ;
- `ROTATION_Y` : à remplir **après** avoir vu les aperçus, si un modèle regarde du mauvais côté (degrés).

In [ ]:
TRIANGLES = 3000
TAILLE = 0.6
RESOLUTION = 256
ROTATION_Y = {}  # ex. {"nirvana_nevermind": 90}

## 4. Chargement des modèles d'IA

In [ ]:
import sys
sys.path.insert(0, "/content/TripoSR")  # pour tsr/ et notre torchmcubes.py
import numpy as np
import torch
import rembg
from PIL import Image
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

modele = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
modele.renderer.set_chunk_size(8192)
modele.to("cuda")
session_rembg = rembg.new_session("u2net")

## 5. Fonctions : détourage, reconstruction, simplification, repère du projet, écriture OBJ

Repère du projet (voir `CLAUDE.md`) : pochette = carré [-0.5, 0.5]² du plan y = 0, **Y vers le haut**,
+Z vers le spectateur. TripoSR produit des maillages **Z vers le haut** : on les tourne de -90° autour de X.

In [ ]:
from scipy.spatial import cKDTree
import fast_simplification


def detourer(chemin):
    """Pochette -> sujet détouré, recentré, sur fond gris (l'entrée attendue par TripoSR)."""
    image = remove_background(Image.open(chemin).convert("RGB"), session_rembg)  # RGBA
    image = resize_foreground(image, 0.85)  # le sujet occupe 85 % de l'image
    a = np.asarray(image).astype(np.float32) / 255.0
    rgb = a[:, :, :3] * a[:, :, 3:4] + (1 - a[:, :, 3:4]) * 0.5
    return Image.fromarray((rgb * 255).astype(np.uint8)), image


def reconstruire(image):
    """TripoSR : image -> maillage trimesh avec couleur par sommet."""
    with torch.no_grad():
        codes = modele([image], device="cuda")
    return modele.extract_mesh(codes, True, resolution=RESOLUTION)[0]


def simplifier(sommets, triangles, couleurs, cible):
    """Décimation quadrique ; les couleurs sont reprises du sommet d'origine le plus proche."""
    if len(triangles) > cible:
        sommets2, triangles2 = fast_simplification.simplify(
            sommets.astype(np.float32), triangles.astype(np.int64), target_reduction=1 - cible / len(triangles))
    else:
        sommets2, triangles2 = sommets, triangles
    _, plus_proche = cKDTree(sommets).query(sommets2)
    return np.asarray(sommets2, np.float64), np.asarray(triangles2, np.int64), couleurs[plus_proche]


def vers_repere_projet(sommets, triangles, degres_y=0.0):
    # Z vers le haut -> Y vers le haut : rotation de -90° autour de X, (x, y, z) -> (x, z, -y).
    s = np.stack([sommets[:, 0], sommets[:, 2], -sommets[:, 1]], axis=1)
    a = np.radians(degres_y)  # correction éventuelle de l'orientation, autour de Y
    s = s @ np.array([[np.cos(a), 0, -np.sin(a)], [0, 1, 0], [np.sin(a), 0, np.cos(a)]])
    # Centré sur la pochette, posé dessus (y min = 0), plus grande dimension = TAILLE.
    mini, maxi = s.min(0), s.max(0)
    s -= [(mini[0] + maxi[0]) / 2, mini[1], (mini[2] + maxi[2]) / 2]
    s *= TAILLE / (maxi - mini).max()
    # Notre rasteriseur élimine les faces arrière : les triangles doivent tourner dans le sens
    # trigonométrique vus de l'extérieur, c'est-à-dire un volume signé positif.
    a_, b_, c_ = (s[triangles[:, k]] for k in range(3))
    if np.einsum("ij,ij->i", a_, np.cross(b_, c_)).sum() < 0:
        triangles = triangles[:, ::-1]
    return s, triangles


def ecrire_obj(chemin, sommets, triangles, couleurs, titre):
    """OBJ lisible par pochettes_ar/scene/maillage.py : « v x y z r g b », faces sans normales
    (notre chargeur calcule les normales lissées)."""
    chemin.with_suffix(".mtl").write_text("newmtl ia\nKd 1 1 1\nKs 0.15 0.15 0.15\nNs 20\n")
    lignes = [f"# {titre} : généré par IA (rembg + TripoSR), {len(triangles)} triangles",
              f"mtllib {chemin.with_suffix('.mtl').name}", "o ia", "usemtl ia"]
    lignes += [f"v {x:.5f} {y:.5f} {z:.5f} {r:.4f} {g:.4f} {b:.4f}" for (x, y, z), (r, g, b) in zip(sommets, couleurs)]
    lignes += [f"f {a + 1} {b + 1} {c + 1}" for a, b, c in triangles]
    chemin.write_text("\n".join(lignes) + "\n")

## 6. Génération de toutes les pochettes

In [ ]:
import time, traceback

resultats = {}
for id_album, a in albums.items():
    titre = f"{a['artiste']} - {a['album']}"
    debut = time.time()
    try:
        image, detouree = detourer(ALBUMS / a["fichier"])
        detouree.save(CACHE / "apercus" / f"{id_album}_detourage.png")
        maillage = reconstruire(image)
        sommets = np.asarray(maillage.vertices)
        triangles = np.asarray(maillage.faces)
        couleurs = np.asarray(maillage.visual.vertex_colors)[:, :3] / 255.0
        avant = len(triangles)
        sommets, triangles, couleurs = simplifier(sommets, triangles, couleurs, TRIANGLES)
        sommets, triangles = vers_repere_projet(sommets, triangles, ROTATION_Y.get(id_album, 0))
        ecrire_obj(CACHE / f"{id_album}.obj", sommets, triangles, couleurs, titre)
        resultats[id_album] = (sommets, triangles, couleurs)
        print(f"OK    {titre:45s} {avant:6d} -> {len(triangles):5d} triangles  ({time.time() - debut:.0f} s)")
    except Exception:
        print(f"ÉCHEC {titre} : l'application gardera le modèle classique")
        traceback.print_exc()

## 7. Vérification visuelle

Pour chaque album : la pochette, le détourage, puis le modèle vu de face (depuis +Z), de côté (depuis +X)
et de dessus. Le modèle doit être **debout** (Y vers le haut) sur les vues de face et de côté.
Sinon : régler `ROTATION_Y` (étape 3) ou retirer l'album, et relancer les étapes 6 à 8.
Un détourage raté (pochette abstraite, sujet trop petit) donne un modèle informe : dans ce cas,
supprimer son `.obj` de `cache/` pour garder le modèle classique.

In [ ]:
import matplotlib.pyplot as plt

def vue(ax, s, t, c, axes, titre):
    """Aperçu rapide (peintre : triangles triés du plus loin au plus proche), pour Colab uniquement."""
    i, j, k = axes  # axes horizontal, vertical, profondeur (vers l'observateur)
    p = s[t]
    ordre = np.argsort(p[:, :, k].mean(1))
    from matplotlib.collections import PolyCollection
    ax.add_collection(PolyCollection(p[ordre][:, :, [i, j]], facecolors=c[t].mean(1)[ordre], edgecolors="none"))
    ax.set_xlim(-0.35, 0.35); ax.set_ylim(-0.05, 0.65) if j == 1 else ax.set_ylim(0.35, -0.35)
    ax.set_aspect("equal"); ax.set_title(titre, fontsize=8); ax.set_xticks([]); ax.set_yticks([])

for id_album, (s, t, c) in resultats.items():
    a = albums[id_album]
    fig, axs = plt.subplots(1, 5, figsize=(16, 3.2))
    axs[0].imshow(Image.open(ALBUMS / a["fichier"])); axs[0].set_title(f"{a['artiste']} - {a['album']}", fontsize=8)
    axs[1].imshow(Image.open(CACHE / "apercus" / f"{id_album}_detourage.png")); axs[1].set_title("détourage", fontsize=8)
    for ax in axs[:2]: ax.axis("off")
    vue(axs[2], s, t, c, (0, 1, 2), "face (depuis +Z)")
    vue(axs[3], np.stack([-s[:, 2], s[:, 1], s[:, 0]], 1), t, c, (0, 1, 2), "côté (depuis +X)")
    vue(axs[4], s, t, c, (0, 2, 1), "dessus (depuis +Y)")
    fig.savefig(CACHE / "apercus" / f"{id_album}_apercu.png", dpi=80, bbox_inches="tight")
    plt.show()

## 8. Téléchargement

Télécharge `cache.zip`. Ensuite, sur ton ordinateur, dans le dossier du projet :

```
unzip ~/Downloads/cache.zip -d .
uv run python -m pochettes_ar.main
```

Les albums qui ont un `cache/<id>.obj` utilisent le modèle IA ; les autres gardent le modèle classique.
Touche `g` dans l'application : basculer entre modèles IA et classiques.

In [ ]:
!cd /content && rm -f cache.zip && zip -qr cache.zip cache && ls -la cache
from google.colab import files
files.download("/content/cache.zip")